In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [2]:
# Đọc file Excel
# df_xlsx = pd.read_excel('BeatAML/raw/beataml2_manuscript_vg_cts.xlsx', sheet_name='Malignant Cell Type Scores')

mut_df = pd.read_csv('../data/raw/beataml/beataml_wes_wv1to4_mutations_dbgap.txt', sep='\t')
clin_df = pd.read_excel('../data/raw/beataml/beataml_wv1to4_clinical.xlsx', sheet_name='summary')
drug_df = pd.read_csv('../data/raw/beataml/beataml_probit_curve_fits_v4_dbgap.txt', sep='\t')
map_df = pd.read_excel('../data/raw/beataml/beataml_waves1to4_sample_mapping.xlsx', sheet_name='sample_map')
drug_gene_df = pd.read_excel('../data/raw/beataml/beataml_drug_families.xlsx', sheet_name='drug_gene')

In [3]:
drug_gene_df.head()

,inhibitor,GeneID,Symbol,description,family,manual_addition,db_min_kd,db_min_adj_tier,targetome_min_assay,targetome_adj_tier,targetome_num_refs_le25,targetome_num_dist_vals_le25
0,ABT-737,572,BAD,BCL2 associated agonist of cell death,NaN,False,NaN,NaN,0.6,TIER_1,2.0,2.0
1,Palbociclib,595,CCND1,cyclin D1,NaN,False,NaN,NaN,2.0,TIER_1,6.0,7.0
2,Flavopiridol,904,CCNT1,cyclin T1,NaN,False,NaN,NaN,3.0,TIER_1,5.0,5.0
3,NF-kB Activation Inhibitor,4790,NFKB1,nuclear factor kappa B subunit 1,NaN,False,0.1,TIER_1,NaN,NaN,NaN,NaN
4,Roscovitine (CYC-202),4790,NFKB1,nuclear factor kappa B subunit 1,NaN,False,0.0,TIER_1,NaN,NaN,NaN,NaN


In [5]:
mut_df

,dbgap_sample_id,capture_type,seqnames,pos_start,pos_end,ref,alt,genotyper,tumor_only,total_reads,...,cdna_position,cds_position,protein_position,amino_acids,codons,existing_variation,refseq,sift,polyphen,exac_af
0,BA2336D,NexteraV1.2,4,106156042,106156043,TC,T,varscan,1,151,...,1804/10166,944/6009,315/2002,S/X,tCc/tc,NaN,NaN,NaN,NaN,NaN
1,BA2336D,NexteraV1.2,4,106190829,106190830,AG,A,varscan,1,74,...,4968/10166,4108/6009,1370/2002,G/X,Ggg/gg,rs756348991,NaN,NaN,NaN,0.000037
2,BA2336D,NexteraV1.2,5,170837543,170837543,C,CTCTG,varscan,1,59,...,1160-1161/1758,859-860/885,287/294,L/LCX,ctc/cTCTGtc,rs758959453&COSM158604,NM_002520.6,NaN,NaN,0.000008
3,BA2643D,NexteraV1.2,11,32456651,32456652,GC,G,varscan,1,51,...,525/3122,240/1554,80/517,L/X,ctG/ct,NaN,NM_024426.4&NM_024424.3,NaN,NaN,NaN
4,BA2643D,NexteraV1.2,2,25457242,25457242,C,T,mutect,1,28,...,2983/4380,2645/2739,882/912,R/H,cGc/cAc,rs147001633&COSM52944&COSM442676,NM_175629.2,deleterious(0),probably_damaging(0.993),0.000593
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11716,BA3100D,Nimblegen Custom Capture,2,25457242,25457242,C,T,mutect,1,213,...,2983/4380,2645/2739,882/912,R/H,cGc/cAc,rs147001633&COSM52944&COSM442676,NM_175629.2,deleterious(0),probably_damaging(0.993),0.000593
11717,BA3100D,Nimblegen Custom Capture,2,25457242,25457242,C,T,varscan,1,213,...,2983/4380,2645/2739,882/912,R/H,cGc/cAc,rs147001633&COSM52944&COSM442676,NM_175629.2,deleterious(0),probably_damaging(0.993),0.000593
11718,BA3100D,Nimblegen Custom Capture,5,170837543,170837543,C,CTCTG,varscan,1,42,...,1160-1161/1758,859-860/885,287/294,L/LCX,ctc/cTCTGtc,rs758959453&COSM158604,NM_002520.6,NaN,NaN,0.000008
11719,BA3100D,Nimblegen Custom Capture,1,115258747,115258747,C,T,mutect,1,524,...,289/4449,35/570,12/189,G/D,gGt/gAt,rs121913237&COSM564,NM_002524.4,deleterious(0),possibly_damaging(0.726),0.000017


In [6]:
# --- BƯỚC 1: XỬ LÝ GENOMIC (MUTATION) ---
print("2. Xử lý Mutation Data...")
# Lấy map giữa DNA ID và Subject ID
dna_map = map_df[['dbgap_subject_id', 'dbgap_dnaseq_sample']].dropna().drop_duplicates()

# Chỉ lấy các mẫu có trong map
valid_samples = dna_map['dbgap_dnaseq_sample']
mut_df = mut_df[mut_df['dbgap_sample_id'].isin(valid_samples)]

# FEATURE SELECTION: Chỉ lấy Top 30 gen hay đột biến nhất (Để test Phase 1)
# (Sau này bạn có thể tăng lên 100 hoặc dùng Elastic Net)
top_genes = mut_df['symbol'].value_counts().head(30).index
mut_df_filtered = mut_df[mut_df['symbol'].isin(top_genes)]

# Pivot: Tạo ma trận (Mẫu x Gen)
genomic_features = pd.crosstab(mut_df_filtered['dbgap_sample_id'], mut_df_filtered['symbol'])
genomic_features = (genomic_features > 0).astype(int) # Chuyển thành 0/1

# Nối với Subject ID
genomic_features = genomic_features.reset_index().merge(dna_map, left_on='dbgap_sample_id', right_on='dbgap_dnaseq_sample')
genomic_features = genomic_features.drop(columns=['dbgap_sample_id', 'dbgap_dnaseq_sample'])
# Gộp theo Subject ID (nếu 1 người có nhiều mẫu thì lấy max - tức là có đột biến ở bất kỳ mẫu nào)
genomic_features = genomic_features.groupby('dbgap_subject_id').max().reset_index()

print(f"   -> Đã tạo ma trận gene: {genomic_features.shape}")

2. Xử lý Mutation Data...
   -> Đã tạo ma trận gene: (685, 31)


In [7]:
genomic_features

,dbgap_subject_id,ASXL1,BCOR,CBL,CEBPA,CSF3R,DNMT3A,EZH2,FLT3,GATA2,...,RAD21,RUNX1,SF3B1,SMC1A,SRSF2,STAG2,TET2,TP53,U2AF1,WT1
0,2000,0,0,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,1
1,2001,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,2003,0,0,0,0,0,0,1,1,0,...,0,1,0,0,0,1,0,0,0,0
3,2004,0,0,0,0,0,0,0,0,0,...,0,0,0,0,1,0,0,0,0,0
4,2005,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
680,2831,0,0,0,0,0,0,0,1,0,...,0,0,0,0,0,0,0,0,0,1
681,2832,0,0,0,0,0,1,0,0,0,...,0,0,0,0,0,0,0,0,0,0
682,2833,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
683,2834,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,0,0


In [8]:
# --- BƯỚC 2: XỬ LÝ CLINICAL ---
print("3. Xử lý Clinical Data...")
# Chọn các trường quan trọng
clin_features = clin_df[['dbgap_subject_id', 'ageAtDiagnosis', 'consensus_sex', 'wbcCount', '%.Blasts.in.BM', 'cohort']].copy()
clin_features.columns = ['dbgap_subject_id', 'age', 'sex', 'wbc', 'blasts', 'cohort'] # Đổi tên

# Chuyển đổi các cột sang dạng số (numeric), các giá trị không hợp lệ sẽ thành NaN
cols_to_fix = ['age', 'wbc', 'blasts']
for col in cols_to_fix:
    clin_features[col] = pd.to_numeric(clin_features[col], errors='coerce')
    
# Xử lý Sex (Male=1, Female=0, Unknown=Nan)
clin_features['sex'] = clin_features['sex'].map({'Male': 1, 'Female': 0})

# QUAN TRỌNG: Groupby theo Patient để loại bỏ trùng lặp (lấy trung bình hoặc max)
# lấy mean cho features số và lấy first/mode cho cohort
clin_features = clin_features.groupby('dbgap_subject_id').agg({
    'age': 'mean',
    'sex': 'mean',
    'wbc': 'mean',
    'blasts': 'mean',
    'cohort': 'first' # Giữ nguyên nhãn cohort
}).reset_index()

# Impute
clin_features[cols_to_fix] = clin_features[cols_to_fix].fillna(clin_features[cols_to_fix].median())
clin_features['sex'] = clin_features['sex'].fillna(0) # or mode

print(f"   -> Đã xử lý clinical: {clin_features.shape}")

3. Xử lý Clinical Data...
   -> Đã xử lý clinical: (805, 6)


In [9]:
clin_features

,dbgap_subject_id,age,sex,wbc,blasts,cohort
0,2000,65.0,1.0,1.600,10.0,Waves1+2
1,2001,68.0,1.0,16.775,60.0,Waves1+2
2,2003,71.0,1.0,45.600,63.0,Waves1+2
3,2004,75.0,1.0,62.870,90.0,Waves1+2
4,2005,52.0,1.0,14.000,50.0,Both
...,...,...,...,...,...,...
800,2831,62.0,0.0,21.700,63.0,Waves3+4
801,2832,62.0,0.0,2.900,26.0,Waves3+4
802,2833,74.0,0.0,43.060,81.0,Waves3+4
803,2834,53.0,1.0,3.810,95.0,Waves3+4


In [10]:
# --- BƯỚC 3: GHÉP NHÃN (DRUG RESPONSE) & TẠO DATASET CUỐI ---
print("4. Ghép nối dữ liệu (Merge)...")
# Chuẩn bị bảng thuốc (Target)
# Dùng AUC thay vì IC50
target_df = drug_df[['dbgap_subject_id', 'inhibitor', 'auc']].copy()

# Tạo nhãn Binary: Sensitive (1) vs Resistant (0) dựa trên Median từng thuốc
# AUC thấp = Nhạy (Thuốc diệt tế bào nhanh)
median_auc = target_df.groupby('inhibitor')['auc'].transform('median')
target_df['response'] = (target_df['auc'] < median_auc).astype(int)

# MERGE 1: Thuốc + Clinical (Inner join để đảm bảo có đủ thông tin)
final_df = target_df.merge(clin_features, on='dbgap_subject_id', how='inner')

# MERGE 2: + Genomic
# Lưu ý: Dùng left join và điền 0. Vì nếu bệnh nhân không có trong bảng mutation -> Coi như không có đột biến top 30 gen này
final_df = final_df.merge(genomic_features, on='dbgap_subject_id', how='left')
# Điền 0 cho gen nếu bệnh nhân không có dữ liệu đột biến (giả sử không đột biến)
gene_cols = [c for c in genomic_features.columns if c != 'dbgap_subject_id']
final_df[gene_cols] = final_df[gene_cols].fillna(0)

# [NEW] Feature: Target Match (Thuốc này có đánh vào gen bệnh nhân đang đột biến không?)
# Tạo từ điển thuốc -> gen targets
drug_target_map = drug_gene_df.groupby('inhibitor')['Symbol'].apply(list).to_dict()

def check_target_match(row):
    drug = row['inhibitor']
    if drug in drug_target_map:
        targets = drug_target_map[drug]
        # Kiểm tra xem có gen target nào nằm trong danh sách gen đột biến của bệnh nhân không
        # (Chỉ xét các gen có trong bảng genomic_features)
        common_genes = [g for g in targets if g in gene_cols and row[g] == 1]
        return 1 if common_genes else 0
    return 0


4. Ghép nối dữ liệu (Merge)...


In [11]:
drug_target_map

{'17-AAG (Tanespimycin)': ['HSP90AA1', 'HSP90AB1'],
 'A-674563': ['CLK1', 'CLK2', 'CLK4', 'DYRK1A', 'DYRK1B'],
 'ABT-737': ['BAD', 'BCL2', 'BCL2L1'],
 'AKT Inhibitor IV': ['AKT1', 'AKT2', 'AKT3'],
 'AKT Inhibitor X': ['AKT1', 'AKT2', 'AKT3'],
 'AMPK Inhibitor': ['PRKAA1', 'PRKAA2'],
 'AST-487': ['ABL1',
  'ABL2',
  'MAP3K20',
  'MAP3K7',
  'FLT3',
  'KIT',
  'MUSK',
  'TIE1',
  'RET',
  'DDR1',
  'MKNK2',
  'CDK7',
  'CDK8',
  'CDK19',
  'CDK13',
  'CDKL2',
  'CDKL3',
  'HIPK3',
  'HIPK4',
  'STK10'],
 'AT7519': ['CDK7',
  'CDK9',
  'CDK13',
  'CDK11B',
  'CDK11A',
  'CDK16',
  'CDK17',
  'CDKL5',
  'GSK3B',
  'CILK1'],
 'AZD1152-HQPA (AZD2811)': ['AURKC', 'AURKB', 'FLT3', 'KIT', 'MAP2K5'],
 'Afatinib (BIBW-2992)': ['EGFR', 'ERBB2', 'ERBB3', 'ERBB4'],
 'Alisertib (MLN8237)': ['AURKA', 'AURKB'],
 'Axitinib (AG-013736)': ['ABL1', 'AURKC', 'KIT', 'PDGFRA', 'PDGFRB', 'KDR'],
 'BEZ235': ['MTOR', 'PIK3CA', 'PIK3CD', 'PIK3CG'],
 'BI-2536': ['PLK3', 'PLK1', 'PLK2'],
 'BMS-345541': ['CHUK',
  '

In [ ]:
# final_df['target_match'] = final_df.apply(check_target_match, axis=1)

# print(f"5. Hoàn tất! Kích thước dataset cuối cùng: {final_df.shape}")
# print(final_df.head())

# # Lưu ra file để bạn dùng train model
# final_df.to_csv('beataml_phase1_dataset.csv', index=False)

5. Hoàn tất! Kích thước dataset cuối cùng: (63395, 39)
   dbgap_subject_id                inhibitor         auc  response   age  sex  \
0              2476     Axitinib (AG-013736)  159.484594         1  27.0  0.0   
1              2476               Crenolanib   69.956422         1  27.0  0.0   
2              2476  Crizotinib (PF-2341066)  146.947463         1  27.0  0.0   
3              2476                Dasatinib  201.043243         0  27.0  0.0   
4              2476                Erlotinib  161.789024         1  27.0  0.0   

      wbc  blasts  ASXL1  BCOR  ...  RUNX1  SF3B1  SMC1A  SRSF2  STAG2  TET2  \
0  20.444    89.0    0.0   0.0  ...    0.0    0.0    0.0    0.0    0.0   0.0   
1  20.444    89.0    0.0   0.0  ...    0.0    0.0    0.0    0.0    0.0   0.0   
2  20.444    89.0    0.0   0.0  ...    0.0    0.0    0.0    0.0    0.0   0.0   
3  20.444    89.0    0.0   0.0  ...    0.0    0.0    0.0    0.0    0.0   0.0   
4  20.444    89.0    0.0   0.0  ...    0.0    0.0    0.0  

In [12]:
final_df

,dbgap_subject_id,inhibitor,auc,response,age,sex,wbc,blasts,cohort,ASXL1,...,RAD21,RUNX1,SF3B1,SMC1A,SRSF2,STAG2,TET2,TP53,U2AF1,WT1
0,2476,Axitinib (AG-013736),159.484594,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1,2476,Crenolanib,69.956422,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
2,2476,Crizotinib (PF-2341066),146.947463,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
3,2476,Dasatinib,201.043243,0,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
4,2476,Erlotinib,161.789024,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63390,2750,Venetoclax,54.410634,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63391,2750,Vismodegib (GDC-0449),286.054744,0,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63392,2750,Volasertib (BI-6727),101.568804,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63393,2750,XMD 8-87,109.579679,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [15]:
# --- BƯỚC 4: (NEW) XỬ LÝ GENE EXPRESSION VỚI HOUSEKEEPING GENES ---
print("6. Xử lý Gene Expression (Relative Expression)...")

# 1. Đọc dữ liệu Gene Expression
print("   -> Đang đọc file expression...")

expr_df = pd.read_csv('BeatAML/raw/beataml_waves1to4_counts_dbgap.txt', sep='\t')

# Xử lý: Set columns index
# Cột 2 là 'display_label' (Symbol), các cột mẫu bắt đầu bằng 'BA'
gene_col = 'display_label'
sample_cols = [c for c in expr_df.columns if c.startswith('BA')]

# Gom nhóm theo Gene Symbol (loại bỏ gen trùng lặp bằng cách cộng counts)
expr_df = expr_df.groupby(gene_col)[sample_cols].sum()
print(f"   -> Đã load expression matrix: {expr_df.shape} (Genes x Samples)")

# Chuyển đổi sang Log2(Counts + 1) để ổn định phương sai cho so sánh
expr_log = np.log2(expr_df + 1)

# 2. Định nghĩa danh sách Housekeeping Genes
HK_genes = ["GPS2", "RPS10", "ZNF91", "FNTA", "PSMC1", "GPS1", "MLH1", 
            "ARAF", "SF3B2", "PSMD2", "SRP14", "GNB1", "HNRNPK", "ARF1", "RPS11"]

valid_HK = [g for g in HK_genes if g in expr_log.index]
print(f"   -> Số lượng Housekeeping genes tìm thấy: {len(valid_HK)}/{len(HK_genes)}")

# 3. Chọn Target Gene để tính toán (nhưng không tạo cột ngay)
target_genes_list = list(set(drug_gene_df['Symbol'].unique())) 
important_genes = [g for g in target_genes_list if g in expr_log.index and g not in valid_HK]
print(f"   -> Số lượng Target genes tìm thấy: {len(important_genes)}")

# 4. Tính toán Relative Expression cho các gene này (Tạo một bảng tra cứu/Lookup Table)
# Thay vì merge thẳng vào final_df, ta tạo một từ điển tra cứu nhanh
# Logic: Gene được coi là biểu hiện cao (1) nếu > hk_mean_series
hk_mean_series = expr_log.loc[valid_HK].mean(axis=0)

# Lấy dữ liệu của các gene quan trọng
important_expr = expr_log.loc[important_genes] # Dạng (Gene x Sample)

# So sánh Vector hóa: Tạo ma trận 0/1 (Gene x Sample)
rel_exp_matrix = (important_expr.gt(hk_mean_series, axis=1)).astype(int)

# Chuyển đổi Sample ID --> Patient ID để dễ tra cứu
# Transpose để Sample là index để dễ merge
rel_exp_T = rel_exp_matrix.T 
rel_exp_T.index.name = 'dbgap_rnaseq_sample'
rel_exp_T = rel_exp_T.reset_index()

# Merge với RNA map để lấy Patient ID
# Tạo rna_map từ map_df (vì rna_map chưa được định nghĩa)
rna_map = (
    map_df[['dbgap_rnaseq_sample', 'dbgap_subject_id']]
    .dropna(subset=['dbgap_rnaseq_sample', 'dbgap_subject_id'])
    .drop_duplicates()
)

patient_exp_df = rel_exp_T.merge(rna_map, on='dbgap_rnaseq_sample', how='inner')
# Group by Patient ID (Lấy Max nếu bệnh nhân có nhiều mẫu)
patient_exp_df = patient_exp_df.groupby('dbgap_subject_id').max()
# Bỏ cột sample sau khi group
if 'dbgap_rnaseq_sample' in patient_exp_df.columns:
    patient_exp_df = patient_exp_df.drop(columns=['dbgap_rnaseq_sample'])

print("   -> Đã tạo bảng tra cứu Expression cho từng bệnh nhân.")

# 5. Tạo Dynamic Feature: "Target_RelExp"
# Biến đổi Dataframe thành Dictionary siêu nhanh để lookup: {(Patient, Gene): Value}
# Stack table lại: Patient | Gene | Value
stacked_exp = patient_exp_df.stack().reset_index()
stacked_exp.columns = ['dbgap_subject_id', 'Symbol', 'Val']
# Tạo hash map
exp_dict = dict(zip(zip(stacked_exp.dbgap_subject_id, stacked_exp.Symbol), stacked_exp.Val))

def get_target_expression(row):
    pid = row['dbgap_subject_id']
    drug = row['inhibitor']
    
    # Tìm gene đích của thuốc này
    if drug not in drug_target_map:
        return 0 # Không biết đích thuốc
    
    targets = drug_target_map[drug]
    
    # Lấy giá trị expression của các gene đích này ở bệnh nhân pid
    # Nếu thuốc có nhiều gene đích -> Lấy trung bình (để biết mức độ biểu hiện chung của pathway)
    values = []
    for t in targets:
        # Tra cứu trong hash map (O(1) complexity)
        val = exp_dict.get((pid, t))
        if val is not None:
            values.append(val)
            
    if not values:
        return 0 # Không có dữ liệu expression của đích
        
    return np.mean(values) # Trả về tỷ lệ gene đích đang biểu hiện cao (vd: 0.5, 1.0)

# Áp dụng vào bảng chính
print("   -> Đang map Target Expression vào Dataset (Dynamic Context)...")
final_df['Target_RelExp'] = final_df.apply(get_target_expression, axis=1)

print(f"6. Hoàn tất! Kích thước dataset tối ưu: {final_df.shape}")


6. Xử lý Gene Expression (Relative Expression)...
   -> Đang đọc file expression...
   -> Đã load expression matrix: (56638, 707) (Genes x Samples)
   -> Số lượng Housekeeping genes tìm thấy: 15/15
   -> Số lượng Target genes tìm thấy: 259
   -> Đã tạo bảng tra cứu Expression cho từng bệnh nhân.
   -> Đang map Target Expression vào Dataset (Dynamic Context)...
6. Hoàn tất! Kích thước dataset tối ưu: (63395, 40)


In [16]:
final_df

,dbgap_subject_id,inhibitor,auc,response,age,sex,wbc,blasts,cohort,ASXL1,...,RUNX1,SF3B1,SMC1A,SRSF2,STAG2,TET2,TP53,U2AF1,WT1,Target_RelExp
0,2476,Axitinib (AG-013736),159.484594,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
1,2476,Crenolanib,69.956422,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,2476,Crizotinib (PF-2341066),146.947463,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
3,2476,Dasatinib,201.043243,0,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
4,2476,Erlotinib,161.789024,1,27.0,0.0,20.444,89.0,Waves1+2,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63390,2750,Venetoclax,54.410634,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63391,2750,Vismodegib (GDC-0449),286.054744,0,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63392,2750,Volasertib (BI-6727),101.568804,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
63393,2750,XMD 8-87,109.579679,1,57.0,1.0,16.710,40.0,Waves3+4,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
# Lưu ra file
final_df.to_csv('../data/interim/beataml_phase1_dataset_v3.csv', index=False)